In [1]:
import os
import shutil
import kagglehub
from pathlib import Path

c:\Users\river\Documents\GitHub\EntrenamientoIA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Descarga y Configuración del Dataset (Cross-Platform)

Esta celda se encarga de preparar el entorno de datos automáticamente, sin importar si usas **Windows, Linux o Mac**.

**¿Qué hace este código?**
1.  **Verificación de Estructura:** Revisa si tienes la carpeta `data/raw` (estándar de Cookiecutter). Si no existe (porque GitHub suele ignorar carpetas vacías o de datos), **la crea automáticamente**.
2.  **Descarga Inteligente:**
    * Utiliza `kagglehub` para bajar la última versión del dataset.
    * Si ya detecta que descargaste los datos previamente en `data/raw/waste_classification`, se salta la descarga para ahorrar tiempo y ancho de banda.
3.  **Gestión de Rutas:** Mueve los archivos desde la caché temporal de Kaggle a nuestra carpeta organizada del proyecto.

**Nota:** No necesitas crear ninguna carpeta manualmente. El script se encarga de todo.

In [2]:
# Definir rutas relativas a partir del directorio base
BASE_DIR = Path("..") 
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
MODELS_DIR = BASE_DIR / "models"

In [3]:
# Crear directorios si no existen
if not DATA_RAW_DIR.exists():
    print(f"La carpeta {DATA_RAW_DIR} no existía. Creándola ahora...")
    DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directorio de datos detectado: {DATA_RAW_DIR}")

La carpeta ..\data\raw no existía. Creándola ahora...


In [6]:
# Descargar el dataset (se descarga en cache primero)
dataset_name = "waste_classification"
local_dataset_path = DATA_RAW_DIR / dataset_name

# Verificamos si ya tenemos el dataset en nuestra carpeta local
if not local_dataset_path.exists():
    print("Iniciando descarga del dataset desde Kaggle...")
    
    # Kaggle descarga por defecto en una carpeta cache del sistema
    cache_path = kagglehub.dataset_download("alistairking/recyclable-and-household-waste-classification")
    cache_path = Path(cache_path) # Convertimos a objeto Path por seguridad
    
    print(f"Moviendo datos desde caché a: {local_dataset_path}...")
    # Copiamos todo el árbol de directorios a nuestra carpeta data/raw
    shutil.copytree(cache_path, local_dataset_path)
    print("¡Descarga y organización completada!")
else:
    print(f"Los datos ya existen en {local_dataset_path}. Se omite la descarga.")


Los datos ya existen en ..\data\raw\waste_classification. Se omite la descarga.


In [7]:
# Buscamos dónde están realmente las imágenes dentro de la carpeta descargada

# Ruta base donde descargaste el dataset
base_images_path = local_dataset_path

# Posible ruta del dataset limpio
clean_dataset_path = base_images_path / "images" / "dataset_final"

# Verificamos si ya existe el dataset limpio (recomendado)
if clean_dataset_path.exists():
    images_path = clean_dataset_path
    print("Usando dataset limpio (dataset_final).")

# Si no existe, intentamos la estructura original automática
else:
    print("dataset_final no encontrado. Buscando estructura original...")

    images_path = base_images_path

    if (images_path / "images").exists():
        images_path = images_path / "images"
    elif (images_path / "waste_classification" / "images").exists():
        images_path = images_path / "waste_classification" / "images"

print(f"\nRuta final de imágenes para entrenamiento: {images_path.resolve()}")

# Verificación adicional: mostrar carpetas detectadas
import os
print("\nCarpetas detectadas en images_path:")
print(os.listdir(images_path))

dataset_final no encontrado. Buscando estructura original...

Ruta final de imágenes para entrenamiento: C:\Users\river\Documents\GitHub\EntrenamientoIA\data\raw\waste_classification\images

Carpetas detectadas en images_path:
['aerosol_cans', 'aluminum_food_cans', 'aluminum_soda_cans', 'cardboard_boxes', 'cardboard_packaging', 'clothing', 'coffee_grounds', 'disposable_plastic_cutlery', 'eggshells', 'food_waste', 'glass_beverage_bottles', 'glass_cosmetic_containers', 'glass_food_jars', 'magazines', 'newspaper', 'office_paper', 'paper_cups', 'plastic_cup_lids', 'plastic_detergent_bottles', 'plastic_food_containers', 'plastic_shopping_bags', 'plastic_soda_bottles', 'plastic_straws', 'plastic_trash_bags', 'plastic_water_bottles', 'shoes', 'steel_food_cans', 'styrofoam_cups', 'styrofoam_food_containers', 'tea_bags']


### Preprocesamiento y "Aumento de Datos" (Data Augmentation)

En esta celda configuramos cómo el modelo "verá" las imágenes. No le pasaremos las fotos tal cual; las transformaremos en tiempo real para que el entrenamiento sea más robusto.

**Conceptos Clave:**
1.  **Normalización (`rescale=1./255`):** Las imágenes son matrices de números del 0 al 255. Las dividimos entre 255 para que queden entre 0 y 1. Esto acelera matemáticamente el aprendizaje de la IA.
2.  **Data Augmentation:** Para evitar que el modelo "memorice" las fotos (overfitting), aplicamos cambios aleatorios en cada época:
    * Girar la imagen (`rotation`).
    * Hacer zoom (`zoom_range`).
    * Moverla horizontal o verticalmente (`shift`).
    * Voltearla (`horizontal_flip`).
    * *Analogía:* Es como entrenar a un portero de fútbol lanzándole balones con diferentes efectos y desde distintos ángulos, no siempre al mismo punto.
3.  **Generadores (`flow_from_directory`):**
    * En lugar de cargar 15,000 imágenes a la RAM (lo cual bloquearía muchas computadoras), estos generadores cargan las imágenes en pequeños lotes (`BATCH_SIZE = 16`) a medida que el modelo las necesita.

**Configuración Técnica:**
* **IMG_SIZE (224, 224):** Tamaño estándar requerido por MobileNetV2.
* **Split 80/20:** Usamos el 80% para entrenar y reservamos un 20% para validar que el modelo realmente está aprendiendo.

In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Configuracion de parametros de entrenamiento

# 224x224 es el tamaño que espera MobileNetV2 por defecto
IMG_SIZE = (224, 224)

# BATCH_SIZE = 16 es conservador y seguro para computadoras con poca VRAM o CPU
BATCH_SIZE = 16

In [9]:
# Creacion del generador 

datagen = ImageDataGenerator(
    rescale=1./255,         # Normalizar pixel values a 0-1
    rotation_range=30,      # Rotar imagen aleatoriamente 30 grados
    width_shift_range=0.2,  # Mover imagen horizontalmente
    height_shift_range=0.2, # Mover imagen verticalmente
    zoom_range=0.3,         # Zoom aleatorio
    horizontal_flip=True,   # Efecto espejo
    validation_split=0.2    # 20% de datos para validación automática
)

In [10]:
# Carga de datos Train y Validation 
print("Cargando set de ENTRENAMIENTO...")
train_gen = datagen.flow_from_directory(
    str(images_path),        # Convertimos la ruta Path a string para evitar errores en TF
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',       # Selecciona el 80%
    shuffle=True             # Mezclar datos para mejor entrenamiento
)

print("Cargando set de VALIDACIÓN...")
val_gen = datagen.flow_from_directory(
    str(images_path),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',     # Selecciona el 20% restante
    shuffle=False            # No es necesario mezclar para validar
)

Cargando set de ENTRENAMIENTO...
Found 12000 images belonging to 30 classes.
Cargando set de VALIDACIÓN...
Found 3000 images belonging to 30 classes.


In [11]:
# Verificacion de clases detectadas
NUM_CLASSES = train_gen.num_classes
print(f"\nTotal de clases detectadas: {NUM_CLASSES}")
print(f"Diccionario de clases: {train_gen.class_indices}")


Total de clases detectadas: 30
Diccionario de clases: {'aerosol_cans': 0, 'aluminum_food_cans': 1, 'aluminum_soda_cans': 2, 'cardboard_boxes': 3, 'cardboard_packaging': 4, 'clothing': 5, 'coffee_grounds': 6, 'disposable_plastic_cutlery': 7, 'eggshells': 8, 'food_waste': 9, 'glass_beverage_bottles': 10, 'glass_cosmetic_containers': 11, 'glass_food_jars': 12, 'magazines': 13, 'newspaper': 14, 'office_paper': 15, 'paper_cups': 16, 'plastic_cup_lids': 17, 'plastic_detergent_bottles': 18, 'plastic_food_containers': 19, 'plastic_shopping_bags': 20, 'plastic_soda_bottles': 21, 'plastic_straws': 22, 'plastic_trash_bags': 23, 'plastic_water_bottles': 24, 'shoes': 25, 'steel_food_cans': 26, 'styrofoam_cups': 27, 'styrofoam_food_containers': 28, 'tea_bags': 29}


### Carga del Modelo Base (Transfer Learning)

Aquí es donde aplicamos la técnica de **Transfer Learning**. En lugar de entrenar una red neuronal desde cero (lo cual requeriría millones de imágenes y mucho poder de cómputo), reutilizamos un modelo que ya es "inteligente".

**¿Qué estamos configurando?**

1.  **MobileNetV2:**
    * Es una arquitectura de red neuronal diseñada específicamente por Google para dispositivos móviles y **IoT** (como nuestra Raspberry Pi). Es ligera y rápida.
2.  **`weights='imagenet'`:**
    * Cargamos el modelo con pesos pre-entrenados en **ImageNet** (un dataset masivo de 1.4 millones de imágenes).
    * *Significado:* Este modelo ya sabe distinguir formas, bordes, texturas, colores y objetos complejos (perros, coches, frutas). Nosotros solo le enseñaremos a diferenciar tipos de basura específicos.
3.  **`include_top=False`:**
    * **Importante:** Cortamos la "cabeza" del modelo (la última capa que clasifica las 1000 clases de ImageNet).
    * Nos quedamos solo con el "cuerpo" (feature extractor) para añadirle después nuestra propia "cabeza" personalizada para nuestras **30 clases de residuos**.
4.  **`base_model.trainable = False` (Congelar el modelo):**
    * Bloqueamos el aprendizaje en estas capas. No queremos que nuestro entrenamiento inicial destruya lo que el modelo ya sabe (detectar bordes, formas, etc.). Solo queremos entrenar las capas nuevas que añadiremos a continuación.

In [12]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2

# Configurar el modelo base pre-entrenado (MobileNetV2)
print("Descargando/Cargando modelo MobileNetV2 (esto puede tardar un poco la primera vez)...")

# Instanciamos el modelo MobileNetV2
base_model = MobileNetV2(
    weights='imagenet',       # Usamos el conocimiento previo de ImageNet
    include_top=False,        # ¡Crucial! Quitamos la capa de clasificación final (la "cabeza")
    input_shape=(224, 224, 3) # Definimos que entrarán imágenes de 224x224 con 3 canales (RGB)
)

# --- CONGELAMIENTO DE CAPAS ---
# "Congelamos" el modelo base. Esto significa que sus pesos (números internos)
# NO cambiarán durante la primera fase del entrenamiento.
# Esto nos permite usarlo solo como un "extractor de características" ultra avanzado.
base_model.trainable = False

print("✅ Modelo base cargado y congelado correctamente.")
print(f"   Estructura de entrada: {base_model.input_shape}")

Descargando/Cargando modelo MobileNetV2 (esto puede tardar un poco la primera vez)...
✅ Modelo base cargado y congelado correctamente.
   Estructura de entrada: (None, 224, 224, 3)


### Construcción de la "Cabecera" del Modelo (El Clasificador)

En este paso, tomamos el modelo base (MobileNetV2) y le añadimos nuestra propia estructura de clasificación al final.

**Analogía:**
* **MobileNetV2 (Base):** Son los **"ojos"**. Ya saben ver líneas, curvas y texturas complejas.
* **Nueva Cabecera (Head):** Es el **"cerebro"** específico para nuestra tarea. Toma lo que ven los ojos y decide: "¿Es esto cartón o vidrio?".

**Capas Clave:**
1.  **`GlobalAveragePooling2D`:**
    * Toma los mapas de características 3D que salen de MobileNetV2 y los "aplasta" promediando sus valores.
    * *Por qué es vital para Raspberry Pi:* Reduce la cantidad de datos enormemente sin perder la información esencial, haciendo el modelo mucho más ligero que si usáramos `Flatten`.
2.  **`Dense(128, relu)`:**
    * Una capa intermedia de neuronas que aprende a combinar las características visuales (ej: "brillante" + "cilíndrico" = "lata").
3.  **`Dropout(0.3)`:**
    * Apaga aleatoriamente el 30% de las neuronas en cada paso del entrenamiento. Esto obliga al modelo a no depender de una sola "pista" y evita el *overfitting* (memorización).
4.  **`Dense(NUM_CLASSES, softmax)`:**
    * La capa final. Tiene tantas neuronas como clases de basura (30).
    * **Softmax:** Convierte los números en **probabilidades** (ej: "80% Plástico, 15% Vidrio, 5% Papel").

In [13]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# Construcción de la nueva cabecera del modelo
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x) # Capa intermedia para aprender patrones complejos
x = Dropout(0.3)(x) # Apaga el 30% de neuronas al azar para evitar memorización (overfitting)

# Capa de salida: NUM_CLASSES debe ser 30 (automático gracias al paso anterior)
output = Dense(NUM_CLASSES, activation='softmax')(x)

# Ensamblamos el modelo final
model = Model(inputs=base_model.input, outputs=output)

print("Arquitectura del modelo finalizada.")
model.summary() # Muestra un resumen para verificar que todo conecta bien

Arquitectura del modelo finalizada.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,425,822 (9.25 MB)

 Trainable params: 167,838 (655.62 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

### Compilación y Entrenamiento (El Aprendizaje)

En esta celda ocurren dos cosas fundamentales:

1.  **Compilación (`model.compile`):** Configuramos las "reglas" con las que el modelo aprenderá.
    * **Optimizador (Adam):** Es el "profesor" que corrige al modelo. Si el modelo se equivoca, Adam ajusta las neuronas para que la próxima vez lo haga mejor.
    * **Learning Rate (`1e-3`):** Es la velocidad de aprendizaje.
        * *0.001* es un estándar seguro. Si es muy alto, el modelo nunca aprende; si es muy bajo, tarda una eternidad.
    * **Loss (`categorical_crossentropy`):** Es la "puntuación de error". El objetivo del modelo es bajar este número a 0. Usamos *categorical* porque tenemos más de 2 tipos de basura.
    * **Metrics (`accuracy`):** Es la "precisión". El porcentaje de aciertos (ej. 80% significa que de 10 fotos, acierta 8).

2.  **Entrenamiento (`model.fit`):** ¡Aquí empieza la acción!
    * El modelo verá todas las imágenes de entrenamiento una y otra vez.
    * **Epochs = 15:** Significa que el modelo repasará el dataset completo 15 veces.
    * **Validation Data:** Al final de cada vuelta (época), se pone a prueba con las imágenes de validación (que nunca ha visto) para ver si realmente está aprendiendo o solo memorizando.

**⚠️ Nota de Rendimiento:**
* Si tienes **GPU (NVIDIA)**, esto será rápido (unos minutos).
* Si usas solo **CPU**, prepárate un café ☕, esto podría tardar bastante más.

In [14]:
import tensorflow as tf
print("Versión de TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Detectada: {gpus}")
else:
    print("⚠️ No se detectó GPU. El entrenamiento correrá en CPU (Lento).")

Versión de TensorFlow: 2.20.0
⚠️ No se detectó GPU. El entrenamiento correrá en CPU (Lento).


In [15]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

# COMPILACIÓN DEL MODELO 
# Definimos cómo va a aprender la red neuronal
print("Compilando el modelo con optimizador Adam...")

model.compile(
    optimizer=Adam(learning_rate=1e-3), # 0.001: Velocidad de aprendizaje inicial estándar
    loss='categorical_crossentropy',    # Obligatorio para clasificación de múltiples clases (>2)
    metrics=['accuracy']                # Queremos monitorear qué tan preciso es (0.0 a 1.0)
)

# ENTRENAMIENTO FITTING 
print("\nIniciando entrenamiento (Fase 1 - Cabecera)...")
print("   Nota: Esto puede tardar dependiendo de tu hardware (CPU vs GPU).")

history = model.fit(
    train_gen,               # Datos de entrenamiento (con data augmentation)
    validation_data=val_gen, # Datos de validación (para examen final de cada época)
    epochs=15,               # Número de vueltas completas al dataset
    verbose=1                # Muestra una barra de progreso animada
)

print("¡Entrenamiento fase 1 completado!")

Compilando el modelo con optimizador Adam...

Iniciando entrenamiento (Fase 1 - Cabecera)...
   Nota: Esto puede tardar dependiendo de tu hardware (CPU vs GPU).
Epoch 1/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 241s 315ms/step - accuracy: 0.5169 - loss: 1.6726 - val_accuracy: 0.6957 - val_loss: 0.9810
Epoch 2/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 230s 307ms/step - accuracy: 0.6518 - loss: 1.1436 - val_accuracy: 0.7250 - val_loss: 0.8829
Epoch 3/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 227s 303ms/step - accuracy: 0.6798 - loss: 1.0196 - val_accuracy: 0.7363 - val_loss: 0.8121
Epoch 4/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 227s 303ms/step - accuracy: 0.6935 - loss: 0.9694 - val_accuracy: 0.7547 - val_loss: 0.7636
Epoch 5/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 226s 301ms/step - accuracy: 0.7103 - loss: 0.9085 - val_accuracy: 0.7607 - val_loss: 0.7407
Epoch 6/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 225s 300ms/step - accuracy: 0.7283 - loss: 0.8657 - val_accuracy: 0.7697 - val_loss: 0.6927
Epoch 7/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 222s 29

### Fine-Tuning (Afinamiento de Precisión)

Hasta ahora, solo hemos entrenado la "cabecera" (el cerebro). El "cuerpo" (MobileNetV2) seguía congelado con lo que aprendió en ImageNet. Ahora vamos a desbloquear una parte del cuerpo para que se adapte mejor a *nuestras* imágenes de basura.

**Estrategia "70/30":**
1.  **Descongelar parcialmente:** Permitimos que las últimas capas del modelo base (el 30% superior) aprendan.
    * *¿Por qué?* Las primeras capas detectan líneas y bordes (útiles para todo). Las últimas capas detectan formas complejas. Queremos adaptar esas formas complejas para que reconozcan botellas aplastadas o cartón arrugado.
2.  **Congelar el 70% inferior:** Mantenemos la base intacta para no perder la estabilidad.

**⚠️ Cambio Crítico: Learning Rate (Tasa de Aprendizaje)**
* Bajamos de `1e-3` a **`1e-5`** (es decir, 100 veces más lento).
* **Razón:** Como el modelo ya sabe bastante, ahora solo queremos hacer **ajustes microscópicos**. Si usamos una tasa alta, destruiríamos el conocimiento previo ("Catastrophic Forgetting").

**Resultado esperado:**
* Esto debería aumentar la precisión final (Accuracy) significativamente.

In [16]:
import tensorflow as tf

print("Iniciando configuración de Fine-Tuning...")

# Descongelamos todo el modelo base primero
base_model.trainable = True

# Decidimos dónde cortar: Congelaremos el 70% inferior
fine_tune_at = int(len(base_model.layers) * 0.7)

# Congelamos las capas anteriores al punto de corte
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"   Total de capas en MobileNetV2: {len(base_model.layers)}")
print(f"   Capas congeladas (70%): {fine_tune_at}")
print(f"   Capas entrenables (30%): {len(base_model.layers) - fine_tune_at}")

# RE-COMPILACIÓN (Obligatorio después de descongelar)
# ¡IMPORTANTE! Usamos un Learning Rate MUY BAJO (1e-5)
# Esto es para hacer "micro-ajustes" sin romper los pesos ya aprendidos.
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nIniciando Fase 2 de Entrenamiento (Fine-Tuning)...")
print("   Nota: Esto tomará más tiempo por época, pero mejorará la precisión.")

# Entrenamiento final
history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20 # Entrenamos por 20 épocas más
)

print("¡Entrenamiento completo finalizado!")

Iniciando configuración de Fine-Tuning...
   Total de capas en MobileNetV2: 154
   Capas congeladas (70%): 107
   Capas entrenables (30%): 47

Iniciando Fase 2 de Entrenamiento (Fine-Tuning)...
   Nota: Esto tomará más tiempo por época, pero mejorará la precisión.
Epoch 1/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 273s 350ms/step - accuracy: 0.6029 - loss: 1.3722 - val_accuracy: 0.7820 - val_loss: 0.6774
Epoch 2/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 263s 351ms/step - accuracy: 0.6836 - loss: 1.0135 - val_accuracy: 0.7807 - val_loss: 0.6546
Epoch 3/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 266s 354ms/step - accuracy: 0.7157 - loss: 0.8915 - val_accuracy: 0.7920 - val_loss: 0.6197
Epoch 4/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 262s 349ms/step - accuracy: 0.7331 - loss: 0.8310 - val_accuracy: 0.7927 - val_loss: 0.6293
Epoch 5/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 257s 343ms/step - accuracy: 0.7376 - loss: 0.7999 - val_accuracy: 0.8147 - val_loss: 0.5598
Epoch 6/20
750/750 ━━━━━━━━━━━━━━━━━━━━ 258s 344ms/step - accuracy: 0.7571 

### Guardado del Modelo (La "Copia Maestra")

Una vez finalizado el entrenamiento, necesitamos guardar todo lo que la IA ha aprendido (los millones de ajustes matemáticos en sus neuronas) en un solo archivo.

**Detalles Técnicos:**
1.  **Formato `.keras`:**
    * Es el **nuevo estándar de TensorFlow 3/Keras**.
    * A diferencia del antiguo `.h5` o de guardar una carpeta llena de archivos (`SavedModel`), este formato es un archivo comprimido (zip) que contiene:
        * La arquitectura (las capas que diseñamos).
        * Los pesos (lo que aprendió).
        * La configuración del optimizador (para poder reanudar el entrenamiento después si queremos).
2.  **Portabilidad:**
    * Este archivo es el que podrías enviarle a un compañero por correo o copiar a la Raspberry Pi para ejecutarlo.

**Nota:** Este archivo es nuestro "backup" principal de alta calidad antes de convertirlo a formatos más ligeros.

In [17]:
import os

# GUARDADO DEL MODELO 

# Aseguramos que el directorio de modelos exista
# Si un compañero borró la carpeta 'models' o no se bajó de git, esto la crea.
if not MODELS_DIR.exists():
    print(f"La carpeta {MODELS_DIR} no existía. Creándola...")
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Definimos la ruta completa
# Usamos la extensión .keras que es el estándar actual más seguro y ligero que .h5
model_save_path = MODELS_DIR / "modelo_residuos_rpi.keras"

print(f"Guardando modelo maestro en: {model_save_path} ...")

# Guardamos el archivo
model.save(model_save_path)

# Confirmación visual con ruta absoluta para evitar confusiones sobre dónde se guardó el modelo
print(f"¡Éxito! El archivo está listo.")
print(f"   Ruta absoluta: {model_save_path.resolve()}")

Guardando modelo maestro en: ..\models\modelo_residuos_rpi.keras ...
¡Éxito! El archivo está listo.
   Ruta absoluta: C:\Users\river\Documents\GitHub\EntrenamientoIA\models\modelo_residuos_rpi.keras


### Conversión a TensorFlow Lite (IoT & Edge AI)

Este paso es **obligatorio** para correr el modelo en la Raspberry Pi de forma fluida. Convertimos el modelo estándar (`.keras`) al formato `.tflite`.

**¿Qué hace la conversión?**
1.  **Traducción:** Reescribe las operaciones matemáticas complejas de TensorFlow a un "lenguaje" simplificado que los procesadores móviles entienden mejor.
2.  **Cuantización (`tf.lite.Optimize.DEFAULT`):**
    * Esta es la parte mágica. Reduce la precisión de los números internos del modelo (de flotantes de 32 bits a 16 o 8 bits).
    * *Resultado:* El archivo pesa 3 o 4 veces menos y corre 2 veces más rápido.
    * *Impacto:* La precisión baja mínimamente (ej. de 95% a 94.8%), pero la velocidad se dispara.

**Resultado:**
Obtendremos un archivo `modelo_residuos_rpi.tflite` que es el único que necesitamos copiar a la Raspberry Pi.

In [19]:
print("Iniciando conversión a TFLite...")

# Cargar el convertidor desde el modelo Keras en memoria
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Aplicar optimizaciones Cuantización
# Esto reduce el tamaño del archivo y acelera la inferencia en la Raspberry Pi
# sin sacrificar casi nada de precisión.
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Realizar la conversión
tflite_model = converter.convert()

# Guardar el archivo .tflite
tflite_save_path = MODELS_DIR / "modelo_residuos_rpi.tflite"

with open(tflite_save_path, "wb") as f:
    f.write(tflite_model)

print(f"Modelo TFLite guardado en: {tflite_save_path}")

# --- COMPARACIÓN DE TAMAÑOS ---
# Esto es meramente para mostrar en el reporte final
keras_size = os.path.getsize(model_save_path) / (1024 * 1024)
tflite_size = os.path.getsize(tflite_save_path) / (1024 * 1024)

print(f"\nReporte de Optimización:")
print(f"   Tamaño Original (.keras): {keras_size:.2f} MB")
print(f"   Tamaño Optimizado (.tflite): {tflite_size:.2f} MB")
print(f"   Reducción de tamaño: {((keras_size - tflite_size) / keras_size) * 100:.2f}% menos")

Iniciando conversión a TFLite...
INFO:tensorflow:Assets written to: C:\Users\river\AppData\Local\Temp\tmpfgft27y7\assets


INFO:tensorflow:Assets written to: C:\Users\river\AppData\Local\Temp\tmpfgft27y7\assets


Saved artifact at 'C:\Users\river\AppData\Local\Temp\tmpfgft27y7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 30), dtype=tf.float32, name=None)
Captures:
  2887270096656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2887270098768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2887270100688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2887270100304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2887270099152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2887270100880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2887270099344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2887270101456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2887270101072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2887270098192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2887